In [ ]:
from enum import Enum, auto
import os
import random
import numpy as np
from collections import deque

import vizdoom as vzd

import torch
import torch.nn as nn
import torch.optim as optim

def resetGame(showGame):
    game = vzd.DoomGame()
    config = {
        "doom_scenario_path": os.path.join(vzd.scenarios_path, "deathmatch3.wad"),
        "doom_map": "map01",
        "doom_skill": 2,

        "available_buttons": [
            vzd.Button.ATTACK,
            vzd.Button.MOVE_RIGHT,
            vzd.Button.MOVE_LEFT,
            vzd.Button.MOVE_BACKWARD,
            vzd.Button.MOVE_FORWARD,
            vzd.Button.TURN_RIGHT,
            vzd.Button.TURN_LEFT,
            vzd.Button.SELECT_WEAPON3,
            vzd.Button.SELECT_WEAPON4
        ],

        # MUST enable richer state
        "available_game_variables": [
            vzd.GameVariable.HEALTH,
            vzd.GameVariable.AMMO2,
            vzd.GameVariable.POSITION_X,
            vzd.GameVariable.POSITION_Y,
            vzd.GameVariable.ANGLE,
            vzd.GameVariable.KILLCOUNT,
            vzd.GameVariable.SELECTED_WEAPON,
            vzd.GameVariable.AMMO3,
            vzd.GameVariable.AMMO4
        ],

        "episode_timeout": 4200,
        "episode_start_time": 1,
        "living_reward": 0.01,
        "mode": vzd.Mode.PLAYER,
        "screen_resolution": vzd.ScreenResolution.RES_1920X1080,
        "render_hud": True,
        
    }

    game.set_config(config)
    game.set_objects_info_enabled(True)
    game.set_window_visible(showGame)

    game.init()
    return game

game = resetGame(False)

numberGenerations = 5
maxDepth = 6
treesPerGeneration = 30
topNumToKeep = 20
topKeptSubtrees = [0, 6, 5, 4, 4, 3]
episodesPerTree = 30
chanceOfEndingEarly = 0.12

crossPollinationsMapping = [
# d0 d1 d2 d3 d4 d5
 [0, 3, 1, 0, 0, 0],  # depth 0
 [ 0,10, 3, 1, 0, 0],  # depth 1
 [ 0, 0,10, 3, 1, 0],  # depth 2
 [ 0, 0, 0,10, 3, 1],  # depth 3
 [ 0, 0, 0, 0,10, 3],  # depth 4
 [ 0, 0, 0, 0, 0,10],  # depth 5
]

remainingTrees = []

allConditions = {
    "lowHealth?",
    "mediumHealth?",
    "highHealth?",
    "lowArmor?",
    "mediumArmor?",
    "highArmor?",
    "lowAmmoCurrent?",
    "mediumAmmoCurrent?",
    "highAmmoCurrent?",
    "overallLowAmmo?",
    "chaingunEquipped?",
    "nearbyEnemy?",
    "recentlyHurt?",
    "healthNearby?",
    "ammoNearby?",
    "armorNearby?",
    "manyEnemiesNearby?",
    "manyEnemies?",
    "noEnemies?",
    "closeRangedEnemy?",
    "highTimeRemaining?",
    "mediumTimeRemaining?",
    "lowTimeRemaining?"
}

allActions = {
    "switchWeapon",
    "fireAndStrafe",
    "directlyFlee",
    "goToHealth",
    "goToAmmo",
    "goToArmor",
    "moveRandom",
    "runAway",
    "chargeIn",
    "kiteDistance"
}

class NodeType(Enum):
    ACTION = auto()
    CONDITION = auto()

class Node():
    def __init__(self, depth, parent, nodeType, value):
        self.depth = depth
        self.parent = parent
        self.type = nodeType
        self.value = value

        self.left = None
        self.right = None

class Tree():
    def __init__(self, maxDepth, probabilityOfEndingEarly):
        self.maxDepth = maxDepth
        self.probabilityOfEndingEarly = probabilityOfEndingEarly

    def buildNode(self, remainingConditions, remainingActions, currentDepth, parentNode):
        randomProbability = random.random()
        if(currentDepth == self.maxDepth or (currentDepth != 0 and randomProbability <= self.probabilityOfEndingEarly) or len(remainingConditions) == 0):
            nodeType = NodeType.ACTION
        else:
            nodeType = NodeType.CONDITION
        

        chosenListToSelect = remainingConditions if nodeType == NodeType.CONDITION else remainingActions
        chosenElement = random.choice(list(chosenListToSelect))
        chosenListToSelect.remove(chosenElement)
        newNode = Node(currentDepth, parentNode, nodeType, chosenElement)
        
        if(nodeType == NodeType.CONDITION):
            newNode.left = self.buildTree(remainingConditions, remainingActions, currentDepth + 1, newNode)

        return newNode

def BuildRandomTree(maxDepth, probEndingEarly, conditions, actions):
    tree = Tree(maxDepth, probEndingEarly)
    tree.root = tree.buildNode(conditions, actions, 0, None)

def RankTreesByScore(treeArray):
    return sorted(treeArray, key = lambda x : x.score)

def clearNodesRecursively(currentNode):
    if(currentNode.isAction):
        return
    currentNode.secureSubtree = False

    clearNodesRecursively(currentNode.left)
    clearNodesRecursively(currentNode.right)

def DoTreeUpkeep(givenTree):
    tree.score = 0
    clearNodesRecursively(tree.root)


for i in range(0, numberGenerations, 1):
    for _ in range(treesPerGeneration - len(remainingTrees)):
        remainingTrees.append(BuildRandomTree(maxDepth, chanceOfEndingEarly, allConditions.copy(), allActions.copy()))
    
    for tree in remainingTrees:
        DoTreeUpkeep(tree)
        for _ in range(episodesPerTree):
            tree.score += RunGame(tree)
    
    remainingTrees = RankTreesByScore(remainingTrees)
    LockSubtrees(remainingTrees)
    remainingTrees = remainingTrees[:topNumToKeep]
    Crossovertrees(remainingTrees)

remainingTrees = RankTreesByScore(remainingTrees)
for i in range(3):
    print("Data about the {i}th best tree:")
    ShowTreeStructure(remainingTrees[i])
